# Model check: same excitation, simulator vs models

Drives the MuJoCo simulator (`make_calibrated_sim`, treated as the real robot) and three models
with **the same commands**, then compares the tip per axis:

| Model | Built by | Inputs it takes | Outputs |
|---|---|---|---|
| One-mode model | Nikoletta | plane pressure differences: x ← −u₃, y ← u₂ − u₄ (no actuator dynamics) | x, y |
| M2 | Ibraheem, `task_space_model.py` | the three segment commands, through dead time 0.5 s + lag 0.6 s | x, y, z |
| F | Ibraheem, `sysid_frf.py` | channels ν_x = −u₃, ν_y = u₂ − u₄, with the identified delay | x, y |

u = command − 4.5 psi. Every model runs **open loop from rest** (free run): it never sees the
simulator's tip, so errors accumulate exactly as they would in a controller's internal model.
All code is in `scripts/compare_models_excitation.py`; this notebook only calls it and plots.
Runs offline in the project environment (`uv run jupyter lab`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import compare_models_excitation as CM
import plot_style as P

P.apply()
%matplotlib inline

## 1 — Choose an excitation

- `"random_steps"`: independent random levels per segment, 1.5–7.5 psi, held 0.5–3 s.
- `"chirp"`: phased sines (0°, 120°, 240°) on S2–S4, 4.5 ± 2.5 psi, frequency swept 0.02 → 1 Hz.
- Or set `CMDS` yourself: any `(n, 3)` array of absolute psi for S2, S3, S4 at 100 Hz, within 0–9.

Seeds used for identification and validation elsewhere: 1–4, 11–14, 21–22, 101. Pick another.

In [ ]:
KIND = "random_steps"     # or "chirp"
SECONDS = 60.0
SEED = 31                 # random_steps only

CMDS = CM.excitation(KIND, SECONDS, seed=SEED)
# CMDS = np.column_stack([...])   # custom excitation: (n, 3) psi, 100 Hz

t = np.arange(len(CMDS)) * CM.DT
fig, ax = plt.subplots(figsize=(P.WIDTH, 3.5))
for j, seg in enumerate(("S2", "S3", "S4")):
    ax.plot(t, CMDS[:, j], **P.line(seg, len(t)))
ax.set(xlabel="time (s)", ylabel="command (psi)", title=f"Excitation: {KIND}")
ax.grid(alpha=0.3); P.legend_outside(ax)
plt.show()

## 2 — Run the simulator and the three models

In [ ]:
u = CMDS - CM.BASE
truth = CM.simulate_truth(CMDS)                  # MuJoCo, seed 0, 5 s settle at 4.5 psi first
sims = {m: CM.SIMULATE[m](u) for m in CM.MODELS}
results = {m: CM.metrics(truth, sims[m]) for m in CM.MODELS}

print(f"{'model':<30}" + "".join(f"{a + ' RMSE (fit)':>22}" for a in "xyz"))
for m in CM.MODELS:
    cells = [f"{v['rmse_mm']:.3f} mm ({v['fit_pct']:5.1f} %)" if v else "no z output" for v in results[m].values()]
    print(f"{CM.LABELS[m]:<30}" + "".join(f"{c:>22}" for c in cells))

**Reading the numbers.** RMSE is in mm. Fit = 100 (1 − ‖model − truth‖ / ‖truth − mean‖): 100 %
is perfect, 0 % is as good as predicting a constant, negative is worse than a constant.

## 3 — Tip per axis

In [ ]:
fig, axs = P.figure(3, 1, sharex=True)
for i, name in enumerate("xyz"):
    ax = axs[i, 0]
    ax.plot(t, truth[:, i], **{**P.REFERENCE, "label": CM.LABELS["truth"]})
    for m in CM.MODELS:
        if not np.isnan(sims[m][:, i]).all():
            ax.plot(t, sims[m][:, i], **P.line(CM.LABELS[m], len(t)))
    ax.set(ylabel=f"tip {name} − settled (mm)", title=f"{name} axis")
    ax.grid(alpha=0.3); P.legend_outside(ax)
axs[-1, 0].set_xlabel("time (s)")
plt.show()

## 4 — Error per axis (model − simulator)

In [ ]:
fig, axs = P.figure(3, 1, sharex=True)
for i, name in enumerate("xyz"):
    ax = axs[i, 0]
    for m in CM.MODELS:
        if not np.isnan(sims[m][:, i]).all():
            ax.plot(t, sims[m][:, i] - truth[:, i], **P.line(CM.LABELS[m], len(t)))
    ax.axhline(0, color="black", lw=1)
    ax.set(ylabel=f"{name} error (mm)", title=f"{name} axis: model − simulator")
    ax.grid(alpha=0.3); P.legend_outside(ax)
axs[-1, 0].set_xlabel("time (s)")
plt.show()

## 5 — Save the data

Writes the commands, the simulator's tip and every model's tip (mm, 100 Hz) to an `.npz` file,
and the metrics to a `.json` file next to it. Load later with `np.load(path)`.

In [ ]:
import json
out = ROOT / "docs" / "figures" / f"model_check_{KIND}_seed{SEED}"
np.savez_compressed(out.with_suffix(".npz"), t=t, cmds=CMDS, truth=truth, **{m: s for m, s in sims.items()})
out.with_suffix(".json").write_text(json.dumps(dict(kind=KIND, seconds=SECONDS, seed=SEED, metrics=results), indent=1))
print("saved", out.with_suffix(".npz"), "and", out.with_suffix(".json"))

## Why the one-mode model is off

It moves in the right direction at the right moments but about 4× too far and with no delay:
its static gain is b/k = 7.7 mm/psi (no gravity stiffness in its derivation), against about
2.0 mm/psi in the simulator (Nikoletta's own measurement, 2.01 mm/psi per segment), and it has
neither the 0.5 s dead time nor the 0.6 s pressure lag. M2 and F were identified from the
simulator's data with those dynamics in, so they follow it to about 0.1 mm.